# New Emotion-Prediction-Dataset — Same-Speaker Persistence

CPU-only Google Colab notebook. Upload the requested dataset file when prompted. Results are saved under `/content/emopredoutputs/` and downloaded automatically as a ZIP.

**Method**

For each source dataset, predict the target speaker’s own most recent prior emotion. If that speaker has not appeared before, fall back to the immediately previous emotion; if unavailable, use that source dataset’s training-set majority. Results are reported separately for DailyDialog, EmoryNLP, and MELD.

**Input to upload:** `DailyDialog_EmoryNLP_MELD_gpt4o_q1.csv`

In [1]:
!pip -q install pandas scikit-learn
from google.colab import files
from pathlib import Path
import pandas as pd, json, shutil
from collections import Counter
from sklearn.metrics import f1_score, accuracy_score
print('Upload DailyDialog_EmoryNLP_MELD_gpt4o_q1.csv')
uploaded=files.upload(); assert uploaded
CSV_PATH=Path(next(iter(uploaded)))

OUT=Path('/content/emopredoutputs/01D_newdataset_same_speaker'); OUT.mkdir(parents=True,exist_ok=True)

Upload DailyDialog_EmoryNLP_MELD_gpt4o_q1.csv


Saving DailyDialog_EmoryNLP_MELD_gpt4o_q1.csv to DailyDialog_EmoryNLP_MELD_gpt4o_q1.csv


In [2]:

df=pd.read_csv(CSV_PATH)
required=['Utterance','Speaker','Emotion','Dialogue_ID','Utterance_ID','Set','dataset']
missing=[c for c in required if c not in df.columns]
assert not missing, f'Missing columns: {missing}'
for c in ['Utterance','Speaker','Emotion','Set','dataset']:
    df[c]=df[c].astype(str).str.strip()
df['Set_norm']=df['Set'].str.lower().replace({'trn':'train','tst':'test','validation':'dev','val':'dev'})
df=df.sort_values(['dataset','Dialogue_ID','Utterance_ID']).reset_index(drop=True)
print(df.groupby(['dataset','Set_norm']).size())

def build_examples(frame):
    out=[]
    for (source,did),g in frame.groupby(['dataset','Dialogue_ID'],sort=False):
        g=g.sort_values('Utterance_ID').reset_index(drop=True)
        for t in range(1,len(g)):
            target=g.iloc[t]; hist=g.iloc[:t]
            out.append({
                'dataset':source,'dialogue_id':str(did),'target_idx':target['Utterance_ID'],
                'split':target['Set_norm'],'target_speaker':target['Speaker'],'gold':target['Emotion'],
                'history_speakers':hist['Speaker'].tolist(),'history_emotions':hist['Emotion'].tolist(),
                'previous_turn_emotion':hist.iloc[-1]['Emotion']
            })
    return out
examples=build_examples(df)

def same_speaker_prev(ex):
    for s,e in zip(reversed(ex['history_speakers']),reversed(ex['history_emotions'])):
        if str(s)==str(ex['target_speaker']): return e
    return None

def metric_block(g):
    return {'n':len(g),'weighted_f1':f1_score(g.gold,g.prediction,average='weighted',zero_division=0),
            'macro_f1':f1_score(g.gold,g.prediction,average='macro',zero_division=0),
            'accuracy':accuracy_score(g.gold,g.prediction)}


dataset      Set_norm
DailyDialog  dev           30
             test         138
             train       1212
EmoryNLP     dev         1120
             test        1218
             train       8434
MELD         dev          706
             test        1585
             train       6142
dtype: int64


In [3]:
rows=[]
for source in sorted(df.dataset.unique()):
    source_ex=[e for e in examples if e['dataset']==source]
    train=[e for e in source_ex if e['split']=='train']
    test=[e for e in source_ex if e['split']=='test']
    assert train, f'No train examples for {source}'
    majority=Counter(e['gold'] for e in train).most_common(1)[0][0]
    for ex in test:
        own=same_speaker_prev(ex)
        pred=own if own is not None else (ex['previous_turn_emotion'] if ex['previous_turn_emotion'] else majority)
        rows.append({**ex,'prediction':pred,'same_speaker_previous':own,
                     'used_fallback':own is None,
                     'train_majority':majority,
                     'is_emotion_shift':None if own is None else ex['gold']!=own})
pred=pd.DataFrame(rows)
pred.to_csv(OUT/'predictions.csv',index=False)
summary=[]
for source,g in pred.groupby('dataset'):
    summary.append({'dataset':source,'subset':'overall',**metric_block(g)})
    for name,val in [('emotion_shift',True),('no_shift',False)]:
        sg=g[g.is_emotion_shift==val]
        if len(sg): summary.append({'dataset':source,'subset':name,**metric_block(sg)})
summary=pd.DataFrame(summary)
summary.to_csv(OUT/'metrics.csv',index=False)
display(summary)
notes={'dataset':'New Emotion-Prediction-Dataset CSV','method':'Same-speaker persistence with previous-turn then source-train-majority fallback','first_turns_excluded':True}
(OUT/'method_notes.json').write_text(json.dumps(notes,indent=2))
zip_path=shutil.make_archive('/content/01D_NewDataset_SameSpeaker_Persistence_results','zip',root_dir=OUT)
files.download(zip_path)

,dataset,subset,n,weighted_f1,macro_f1,accuracy
0,DailyDialog,overall,125,0.443577,0.252283,0.464000
1,DailyDialog,emotion_shift,59,0.000000,0.000000,0.000000
2,DailyDialog,no_shift,53,1.000000,1.000000,1.000000
3,EmoryNLP,overall,1143,0.317505,0.295204,0.319335
4,EmoryNLP,emotion_shift,645,0.000000,0.000000,0.000000
5,EmoryNLP,no_shift,315,1.000000,1.000000,1.000000
6,MELD,overall,1465,0.384263,0.270580,0.384983
7,MELD,emotion_shift,741,0.000000,0.000000,0.000000
8,MELD,no_shift,486,1.000000,1.000000,1.000000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>